# Model-based RL: learn the world, then plan in it

_Reinforcement Learning_

**Instead of only learning from real trial-and-error, learn a model of how the world works and plan against it — so each real sample teaches you far more.**

Model-free RL learns the answer (a value or a policy) directly from experience and never builds a model of the
       world. Model-based RL learns the world &mdash; an estimate $\hat P$ of the dynamics and $\hat R$ of the
       reward &mdash; and then plans against that learned world to get the answer. One real experience can then be reused
       many times: replay it through the model, look ahead with it, dream new trajectories from it.

       Why bother? Sample efficiency. Model-free RL's great weakness is that it needs an enormous number of real
       interactions, because each real step updates only one value. With a model you can generate imagined experience for
       free and plan ahead, so each real sample teaches you much more. That is exactly why model-based methods dominate when
       real samples are precious (robots) or where look-ahead is decisive (games).

---

This notebook is a practice scaffold for the **Model-based RL: learn the world, then plan in it** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy (tabular Dyna-Q on a gridworld)

We implement **Dyna-Q**: ordinary Q-learning plus a learned model of the world that we replay for free. On the classic 6x9 Dyna maze we compare plain Q-learning (`n=0` planning steps) against Dyna-Q with 5 and 50 planning steps, measuring how many **real** environment steps each needs to find a near-optimal policy.

### Step 1 — Define the gridworld and its helpers

This is the classic Sutton & Barto Dyna maze: a 6x9 grid with a start at `(2,0)`, a goal at `(0,8)`, and a handful of wall cells. The dynamics are deterministic, with a reward of `+1` only on reaching the goal. We define small helpers: `free` (is a cell walkable), `sidx` (flatten `(row,col)` to a state index), `step` (the transition), and `argmax_rand` (greedy action with random tie-breaking, which matters early when all Q-values are equal).

In [ ]:
import numpy as np

# Classic Dyna maze (Sutton & Barto): 6x9 grid, start (2,0), goal (0,8).
ROWS, COLS = 6, 9
WALLS = {(1, 2), (2, 2), (3, 2), (4, 5), (0, 7), (1, 7), (2, 7)}
START, GOAL = (2, 0), (0, 8)
ACTS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}    # up, down, left, right
nA = 4

def free(s):
    return 0 <= s[0] < ROWS and 0 <= s[1] < COLS and s not in WALLS

def sidx(s):
    return s[0] * COLS + s[1]

def step(s, a):                                  # deterministic; +1 only at goal
    dr, dc = ACTS[a]
    ns = (s[0] + dr, s[1] + dc)
    if not free(ns):
        ns = s                                   # walls / borders bounce back
    if ns == GOAL:
        return ns, 1.0, True
    return ns, 0.0, False

def argmax_rand(row, rng):                        # break ties at random (crucial early on)
    cand = np.flatnonzero(row == row.max())
    return int(cand[rng.integers(len(cand))])

### Step 2 — The Dyna-Q learning loop

Each real step does three things: a **real** Q-learning backup from the transition just observed; **learning the model** by remembering `(s,a) -> (r, s', done)`; and then `n_plan` **planning** backups on transitions sampled from that learned model. The crucial point is that the planning backup is the *identical* Q-update — it just draws its transition from memory instead of the environment, costing zero real steps. Setting `n_plan=0` recovers plain Q-learning.

In [ ]:
def dyna_q(n_plan, episodes=50, alpha=0.1, gamma=0.95, eps=0.1, seed=0, cap=2000):
    rng = np.random.default_rng(seed)
    Q = np.zeros((ROWS * COLS, nA))
    model = {}                                    # learned model: (s_idx,a) -> (r, s'_idx, done)
    steps_per_ep = []

    for _ in range(episodes):
        s = START
        n_steps = 0
        done = False
        while not done and n_steps < cap:
            si = sidx(s)
            # epsilon-greedy action selection.
            if rng.random() < eps:
                a = int(rng.integers(nA))
            else:
                a = argmax_rand(Q[si], rng)
            ns, r, done = step(s, a)
            nsi = sidx(ns)

            # ----- REAL update (ordinary Q-learning backup) -----
            target = r + (0 if done else gamma * Q[nsi].max())
            Q[si, a] += alpha * (target - Q[si, a])

            # ----- learn the model: remember what really happened -----
            model[(si, a)] = (r, nsi, done)

            # ----- PLAN: n updates on transitions sampled FROM THE MODEL (no real steps) -----
            if n_plan:
                keys = list(model.keys())
                for _ in range(n_plan):
                    (psi, pa) = keys[rng.integers(len(keys))]
                    pr, pnsi, pdone = model[(psi, pa)]
                    pt = pr + (0 if pdone else gamma * Q[pnsi].max())
                    Q[psi, pa] += alpha * (pt - Q[psi, pa])    # SAME backup, imagined transition

            s = ns
            n_steps = n_steps + 1
        steps_per_ep.append(n_steps)
    return steps_per_ep

### Step 3 — Compare planning depths by real-step cost

We average the learning curves over 20 seeds, then ask, for each planning depth, how many real environment steps it takes to first hit a near-optimal episode (`<= 20` steps; the optimal path is 14). Plain Q-learning needs thousands of real steps, while Dyna-Q with planning gets there in far fewer — the sample-efficiency payoff of learning and replaying a model.

In [ ]:
def avg(n_plan, seeds=20):                         # average steps/episode over 20 seeds
    curves = [dyna_q(n_plan, seed=s) for s in range(seeds)]
    return np.array(curves).mean(axis=0)

def steps_until_good(curve, thresh=20):            # cumulative REAL steps to first near-optimal episode
    cum = 0
    for ep, v in enumerate(curve):
        cum += v
        if v <= thresh:
            return ep + 1, int(cum)
    return None, int(cum)

for n in (0, 5, 50):
    ep, real = steps_until_good(avg(n))
    tag = "Q-learning" if n == 0 else f"Dyna-Q (n={n}) "
    print(f"{tag}: near-optimal by episode {ep:2d}  after {real:5d} REAL env steps")
# Optimal path length = 14 steps.
# Q-learning   : near-optimal by episode 28  after  4409 REAL env steps
# Dyna-Q (n=5) : near-optimal by episode  6  after  1156 REAL env steps
# Dyna-Q (n=50): near-optimal by episode  4  after   991 REAL env steps
# Planning lets Dyna reach a good policy with ~4x FEWER real interactions.

## Visualize the data & results

_Does planning on a learned model let Dyna-Q reach the goal in fewer REAL environment steps than plain Q-learning? Plot steps-to-goal per episode for n=0 (plain Q-learning) vs n=5 vs n=50 planning steps._

### Step 1 — Rebuild the maze and the Dyna-Q learner

So the plotting cell is self-contained, we recreate the same 6x9 maze, helpers, and `dyna_q` function. The logic is identical to the reference implementation: real Q-backup, then learn the model, then `n_plan` imagined backups per real step.

In [ ]:
import numpy as np

# Classic 6x9 Dyna maze; start (2,0), goal (0,8); deterministic; +1 only at goal.
ROWS, COLS = 6, 9
WALLS = {(1, 2), (2, 2), (3, 2), (4, 5), (0, 7), (1, 7), (2, 7)}
START, GOAL = (2, 0), (0, 8)
ACTS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
nA = 4

def free(s):
    return 0 <= s[0] < ROWS and 0 <= s[1] < COLS and s not in WALLS

def sidx(s):
    return s[0] * COLS + s[1]

def step(s, a):
    dr, dc = ACTS[a]
    ns = (s[0] + dr, s[1] + dc)
    if not free(ns):
        ns = s
    if ns == GOAL:
        return ns, 1.0, True
    return ns, 0.0, False

def argmax_rand(row, rng):
    cand = np.flatnonzero(row == row.max())
    return int(cand[rng.integers(len(cand))])

def dyna_q(n_plan, episodes=50, alpha=0.1, gamma=0.95, eps=0.1, seed=0, cap=2000):
    rng = np.random.default_rng(seed)
    Q = np.zeros((ROWS * COLS, nA))
    model = {}
    out = []
    for _ in range(episodes):
        s = START
        t = 0
        done = False
        while not done and t < cap:
            si = sidx(s)
            if rng.random() < eps:
                a = int(rng.integers(nA))
            else:
                a = argmax_rand(Q[si], rng)
            ns, r, done = step(s, a)
            nsi = sidx(ns)
            # real backup
            Q[si, a] += alpha * (r + (0 if done else gamma * Q[nsi].max()) - Q[si, a])
            # learn model
            model[(si, a)] = (r, nsi, done)
            # plan
            if n_plan:
                keys = list(model.keys())
                for _ in range(n_plan):
                    psi, pa = keys[rng.integers(len(keys))]
                    pr, pnsi, pdone = model[(psi, pa)]
                    Q[psi, pa] += alpha * (pr + (0 if pdone else gamma * Q[pnsi].max()) - Q[psi, pa])
            s = ns
            t = t + 1
        out.append(t)
    return out

### Step 2 — Average over seeds and print the curves

We average each planning depth's steps-per-episode over 20 seeds and print the resulting curves. As the comments note, plain Q-learning (`n=0`) only settles near 18 steps after about episode 28, while `n=5` and `n=50` reach near-optimal (~17, against the optimal 14) within a handful of episodes — the curves you would plot to see model-based RL's edge.

In [ ]:
def avg(n, seeds=20):
    curves = [dyna_q(n, seed=s) for s in range(seeds)]
    return np.array(curves).mean(axis=0)

for n in (0, 5, 50):
    c = avg(n)
    print(f"n={n:2d}", [int(round(x)) for x in c])
# n= 0 -> settles ~18 steps/ep only after ~episode 28 (slow; needs many real steps)
# n= 5 -> at/near optimal (~17) by episode ~6
# n=50 -> at/near optimal (~17) by episode ~4
# Optimal path length is 14. Dyna reaches good policy in far fewer REAL steps.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In Dyna-Q, what is the only difference between a "real" Q-update and a "planning" Q-update, and why does adding planning updates make the agent reach a good policy in fewer REAL environment steps?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write both updates: both are $Q(s,a)\leftarrow Q(s,a)+\alpha[\,r+\gamma\max_{a'}Q(s',a')-Q(s,a)\,]$. — _Dyna deliberately uses the identical backup for both, so they accumulate into the same $Q$._
- Note where $(s,a,r,s')$ comes from: the real update uses a transition just observed in the environment; the planning update uses one SAMPLED from the learned model $\hat P,\hat R$. — _That is the whole distinction &mdash; real vs imagined source of the transition._
- Each planning update costs zero real steps but still propagates reward information across states. — _So $n$ planning updates per real step multiply how much each real sample teaches._

**Answer:** The update rule is identical; only the SOURCE of the transition differs (real environment vs the learned model). Planning updates spread reward information for free, so the policy becomes good in far fewer real interactions &mdash; the sample-efficiency advantage of model-based RL.

</details>

**Problem 2.** In MCTS, a node $s$ has been visited $N(s)=100$ times. Action "left" has $Q=0.6,\ N(s,\text{left})=80$; action "right" has $Q=0.5,\ N(s,\text{right})=4$. With $c=1$, which does UCT select?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the UCT score $Q(s,a)+c\sqrt{\ln N(s)/N(s,a)}$ for each. Here $\ln 100\approx 4.605$. — _UCT adds an exploration bonus to the exploit value $Q$._
- Left: $0.6 + 1\cdot\sqrt{4.605/80}=0.6+\sqrt{0.0576}=0.6+0.240=0.840$. — _Left is well-explored, so its bonus is small._
- Right: $0.5 + 1\cdot\sqrt{4.605/4}=0.5+\sqrt{1.151}=0.5+1.073=1.573$. — _Right was tried only 4 times, so its bonus is large._

**Answer:** UCT selects "right" ($1.573 \gt 0.840$). Even though "left" has the higher mean value, "right" is under-explored, so the bonus dominates &mdash; exactly the exploration UCT is designed to force before committing.

</details>

**Problem 3.** Why must model-based RL keep its imagined rollouts SHORT, and what specifically goes wrong if it plans 50 steps ahead with a slightly-wrong model?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the model is an estimate $\hat P\neq P$ with some per-step error. — _It was fit from finite data, so it is never exact._
- In a multi-step rollout, the next state feeds the model again, so errors compound: a small step-1 error puts you in a slightly wrong state, whose prediction is more wrong, and so on. — _Errors multiply along the trajectory rather than staying bounded._
- After many steps the imagined trajectory can be physically meaningless, so any plan built on it is unreliable. — _The planner optimizes confidently against a fantasy._

**Answer:** Because model errors COMPOUND: each step is predicted from an already-imperfect state, so a 50-step rollout can drift into nonsense and the planner produces a confident but wrong plan. The fixes are short rollouts, frequent re-grounding on real data (as Dyna does with single-step samples), and modeling uncertainty so the planner distrusts regions the model has not seen.

</details>